In [ ]:
import os

import psycopg2
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

In [ ]:
conexao = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)

schemas = pd.read_sql("SELECT schema_name FROM information_schema.schemata ORDER BY schema_name;", conexao)
schemas

In [7]:
tabelas = pd.read_sql(
    "SELECT table_name, table_type FROM information_schema.tables WHERE table_schema = 'Pacientes' ORDER BY table_name;",
    conexao,
)
tabelas

C:\Users\Admin\AppData\Local\Temp\ipykernel_12676\2956016678.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tabelas = pd.read_sql(


,table_name,table_type


## Tabela `Pacientes.pacientes`

A tabela ainda não existe — a célula abaixo cria a tabela dentro do schema `Pacientes`.

In [ ]:
cur = conexao.cursor()
cur.execute("""
    CREATE TABLE IF NOT EXISTS "Pacientes".pacientes (
        id SERIAL PRIMARY KEY,
        nome VARCHAR(255) NOT NULL,
        cpf VARCHAR(11) UNIQUE NOT NULL,
        data_nascimento DATE NOT NULL,
        telefone VARCHAR(20),
        email VARCHAR(255),
        endereco VARCHAR(255)
    );
""")
conexao.commit()
cur.close()

### Create — inserir paciente

In [ ]:
novo_paciente = {
    "nome": "Maria da Silva",
    "cpf": "12345678900",
    "data_nascimento": "1990-05-20",
    "telefone": "11999998888",
    "email": "maria.silva@example.com",
    "endereco": "Rua das Flores, 123",
}

cur = conexao.cursor()
cur.execute(
    """
    INSERT INTO "Pacientes".pacientes (nome, cpf, data_nascimento, telefone, email, endereco)
    VALUES (%(nome)s, %(cpf)s, %(data_nascimento)s, %(telefone)s, %(email)s, %(endereco)s)
    RETURNING id;
    """,
    novo_paciente,
)
novo_id = cur.fetchone()[0]
conexao.commit()
cur.close()
novo_id

### Read — listar / buscar pacientes

In [ ]:
# lista todos os pacientes
pacientes = pd.read_sql('SELECT * FROM "Pacientes".pacientes ORDER BY id;', conexao)
pacientes

In [ ]:
# busca um paciente específico por id
paciente = pd.read_sql(
    'SELECT * FROM "Pacientes".pacientes WHERE id = %(id)s;',
    conexao,
    params={"id": novo_id},
)
paciente

### Update — atualizar paciente

In [ ]:
dados_atualizados = {
    "id": novo_id,
    "telefone": "11988887777",
    "email": "maria.silva.novo@example.com",
}

cur = conexao.cursor()
cur.execute(
    """
    UPDATE "Pacientes".pacientes
    SET telefone = %(telefone)s,
        email = %(email)s
    WHERE id = %(id)s;
    """,
    dados_atualizados,
)
conexao.commit()
cur.close()

### Delete — remover paciente

In [ ]:
cur = conexao.cursor()
cur.execute(
    'DELETE FROM "Pacientes".pacientes WHERE id = %(id)s;',
    {"id": novo_id},
)
conexao.commit()
cur.close()